<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/01_mental_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 01 — From Function Calling to Agents

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

> **Where you are** — course 2 of the path, right after *Úvod do GenAI v Pythone*.
> - **You can already:** call a model with `client.responses.create(...)`, hand-write a tool
>   schema, and run your own function-calling loop (the function-calling chapter of that course).
> - **New here:** ADK's four building blocks — `Agent`, `Runner`, `Event`, `Session` — and
>   your first taste of `async`.

At the end of the previous course we made a promise: there is a ladder above function calling. **MCP** standardized how tools are offered to models, and Google's **Agent Development Kit (ADK)** climbed one level higher — into *orchestration*, where even calling another agent is just a function call. This course is that promised next rung.

And here is the claim this notebook will prove: **you already built an agent in the previous course — you just didn't call it that.**

**What we'll do:**
1. Re-run *your* weather assistant from the previous course — with exactly one changed line.
2. Look at its limits with a builder's eye.
3. Meet ADK's four building blocks and map each onto code you already wrote.
4. Build the same agent in ADK — first without a tool, then with one.

**Runs in:** Google Colab or local Python 3.10+. &nbsp; **Cost:** well under $0.01.

# Setup

## Install dependencies

Same ritual as the previous course — one pip line.

In [1]:
!pip install -q google-adk==2.7.1 openai==2.54.0 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## One New Key — OpenRouter

In the previous course everything ran on your `OPENAI_API_KEY`. That key opens exactly one vendor — and this course's whole point is that ADK is **vendor-neutral**: in Module 04 the *same agent* will run on OpenAI, Anthropic, Google, Qwen and Meta models. So we use **OpenRouter** ([openrouter.ai/keys](https://openrouter.ai/keys)): one key, one small balance, every vendor, at (nearly) the providers' own prices. All of Part 1 costs a couple of dollars.

The setup ritual is the one you already know:

- **Colab:** 🔑 icon in the left sidebar → *Add new secret* → Name: `OPENROUTER_API_KEY` → Value: your key → enable notebook access. (Your `OPENAI_API_KEY` can stay there; they don't conflict.)
- **Local:** put `OPENROUTER_API_KEY=sk-or-...` in a `.env` file next to the notebook — or just paste the key when the cell prompts you.

In [2]:
import os

# Try Colab secrets first, then .env, then prompt.
OPENROUTER_API_KEY = None

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Tip for Colab users: Go to 🔑 (left sidebar) → Add new secret → Name: OPENROUTER_API_KEY")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY and OPENROUTER_API_KEY.strip(), "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Cheap + fast model for all vendor-agnostic demos. Swap it to try another provider.
MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


# The Agent You Already Built

In the function-calling chapter of the previous course you wrote a weather assistant:

- `get_current_weather(location, format)` — a plain Python function,
- a hand-written `tools` schema — 34 lines of JSON describing that function to the model,
- `available_functions` — your dispatch dictionary,
- `chat_with_function_execution(...)` — the five-step loop that let the model *use* the function.

Let's rebuild its essence in two cells. **The only surprise is one line:** `base_url`. OpenRouter speaks the same Responses API your code already speaks — you point the `OpenAI` client at a different address, and the model string gains a vendor prefix (`openai/gpt-5.6-luna` = "OpenAI's gpt-5.6-luna, please").

In [3]:
from openai import OpenAI
import json

# The one changed line vs the previous course: base_url points at OpenRouter.
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
OPENAI_MODEL = "openai/gpt-5.6-luna"   # vendor prefix: OpenAI's model, served via OpenRouter

# Your function from the previous course — same signature, same docstring style.
# One change: deterministic data instead of random, so re-runs are comparable.
FAKE_WEATHER = {
    "Bratislava": ("Sunny", 21.0),
    "Prague":     ("Cloudy", 14.0),
    "Munich":     ("Rainy", 11.0),
}

def get_current_weather(location: str, format: str) -> dict:
    """Get the current weather for a location.

    Args:
        location (str): The city, e.g. "Bratislava" or "Prague".
        format (str): The temperature format, either 'celsius' or 'fahrenheit'.

    Returns:
        dict: location, temperature, unit and condition (mock data).
    """
    city = location.split(",")[0].strip()
    condition, celsius = FAKE_WEATHER.get(city, ("Unknown", None))
    if celsius is None:
        return {"location": location, "condition": "Unknown", "error": f"No data for {location}."}
    temp = celsius if format == "celsius" else round(celsius * 9 / 5 + 32, 1)
    return {"location": location, "temperature": temp, "unit": format, "condition": condition}

# Your hand-written schema from the previous course — every field written by you, by hand.
tools = [{
    "type": "function",
    "name": "get_current_weather",
    "description": "Get the current weather in a given location. Use this when the user asks "
                   "about current or present weather conditions.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string",
                         "description": "The city, e.g. 'Bratislava' or 'Prague'."},
            "format": {"type": "string", "enum": ["celsius", "fahrenheit"],
                       "description": "The temperature unit to use."},
        },
        "required": ["location", "format"],
    },
}]

# Your dispatch dictionary: schema name -> actual Python function.
available_functions = {"get_current_weather": get_current_weather}

print("✅ Your setup from the previous course, rebuilt. One changed line: base_url.")


✅ Your setup from the previous course, rebuilt. One changed line: base_url.


Now the loop. This is your `chat_with_function_execution` from the previous course, condensed — same five steps, same order. One detail to watch: your original loop sent the tool result back as a *fake user message* (a shortcut, kept here honestly). Remember it — it comes up again in a minute.

In [4]:
def chat_with_function_execution(messages, tools, available_functions):
    """Your five-step loop from the previous course, condensed. Same steps, same order."""
    # Step 1: initial API call — the model sees the schemas and decides.
    response = client.responses.create(model=OPENAI_MODEL, input=messages,
                                       tools=tools, tool_choice="auto")
    function_calls = [item for item in response.output
                      if getattr(item, "type", None) == "function_call"]
    if not function_calls:
        return response.output_text  # no tool needed — plain answer

    for tool_call in function_calls:
        # Step 2: extract what the model wants ('arguments' is a JSON *string* — parse it).
        function_arguments = json.loads(tool_call.arguments)
        print(f"[model asked for] {tool_call.name}({function_arguments})")
        # Step 3: execute the real Python function via your dispatch dict.
        function_result = available_functions[tool_call.name](**function_arguments)
        print(f"[function returned] {function_result}")
        # Step 4: send the result back. (Notebook 5's shortcut, kept honestly: we smuggle
        # the result in as a fake *user* message instead of a proper tool-result message.)
        messages.append({"role": "user",
                         "content": f"Function {tool_call.name} returned: {json.dumps(function_result)}"})

    # Step 5: final call — the model writes the answer using the result.
    final_response = client.responses.create(model=OPENAI_MODEL, input=messages)
    return final_response.output_text

messages = [
    {"role": "system", "content": "You are a helpful weather assistant. Use the provided "
                                  "functions to get accurate weather data."},
    {"role": "user", "content": "What's the weather in Bratislava right now? Use Celsius."},
]
answer = chat_with_function_execution(messages, tools, available_functions)
print(f"\n🤖 {answer}")


[model asked for] get_current_weather({'location': 'Bratislava', 'format': 'celsius'})
[function returned] {'location': 'Bratislava', 'temperature': 21.0, 'unit': 'celsius', 'condition': 'Sunny'}



🤖 Bratislava is currently **21°C and sunny**.


### 🔍 What just happened?

Read the printed lines against the five steps:

- `[model asked for] get_current_weather({...})` — the model saw your schema and *decided* to call the function.
- `[function returned] {...}` — *your Python* executed it; the model never runs code.
- The final sentence — the model wrote the answer using the returned data.

That decide → execute → answer cycle **is** an agent. You built one months ago.

### 🎯 Mini-task: one line, new vendor

In the setup cell above, change `OPENAI_MODEL` to `"anthropic/claude-haiku-4.5"` and re-run both cells. Your previous-course code just ran on a **Claude** model — that is what "one key, every vendor" buys you. (Switch it back before you continue.)

# The Ceiling of the Hand-Rolled Loop

The loop works. But look at it with a builder's eye:

- **One round only.** If the model wanted a second tool call after seeing the first result, the loop wouldn't notice.
- **The fake user message.** The API has a proper tool-result message type; the shortcut is fine for learning, wrong for production.
- **You rewrite this loop for every project** — and no two hand-rolled loops are alike, so nothing is reusable.
- **No memory.** `messages` lives in one cell. A user coming back tomorrow? Your problem.
- **No observability.** Your `print()`s are the only window into what happened.

ADK is what you get when this loop is written once, properly, by a team that hit all five problems.

## The Map: Your Loop → ADK

Every piece of your loop maps onto an ADK concept. This table is the heart of the module — the rest of the notebook walks it row by row:

| Your code from the previous course | ADK | The upgrade |
|---|---|---|
| Hand-written `tools` schema | `tools=[get_current_weather]` | ADK **generates the schema from the docstring + type hints** you already write |
| `available_functions` dict | built-in dispatch | name→function routing, argument parsing, error wrapping |
| `system` message | `instruction=` | plus templating from state (M05) |
| Your Steps 1–5 loop | `Runner` | loops until done, any number of tool rounds |
| `messages` list | `Session` | stored, multi-user, survives restarts |
| Your `print()`s | `Event` stream | every step is a typed, inspectable object |

Nothing in ADK is magic — you have written the naive version of every row by hand.

# What Is LiteLLM? (One Thing to Clear Up First)

ADK is Google's framework, and out of the box it speaks to exactly one model family: Gemini. We want the *same code* to run on GPT, Claude, Qwen — whatever is cheapest next year. The problem: every provider's API differs slightly. Different URL, different JSON shapes, different tool formats.

Two cells ago you dodged that with the `base_url` trick — but that trick only works for providers that imitate OpenAI's API. **LiteLLM is that trick, generalized**: an open-source library that translates one model string — `"openrouter/openai/gpt-5.6-luna"` — into the right URL, request shape and response shape, for 100+ providers. It is not part of ADK, and not made by Google.

**`LiteLlm`** (capital L, from `google.adk.models.lite_llm`) is the small **adapter class** that connects the two sides: ADK expects a "model object" it can hand messages to, and `LiteLlm(model="...")` implements that interface by calling the litellm library underneath.

So, to answer the question you probably have: *we are not changing any settings inside ADK, and we are not writing our own model code.* We build one adapter object and plug it into the agent's `model=` slot. That is the whole trick.

One naming gotcha before you see it in code. In the bridge you wrote `openai/gpt-5.6-luna` — OpenRouter's own name for the model. LiteLLM needs one more prefix, to know *which provider to route through*:

```
openrouter / openai/gpt-5.6-luna
   │            │
   │            └── the model, named the way OpenRouter names it (vendor/model)
   └── the provider LiteLLM should route through
```

Now the imports — plus a few lines that silence noisy warnings (the comments say why).

In [5]:
import os
# Silence harmless import-time warnings before heavy imports.
import warnings, io, contextlib
warnings.filterwarnings("ignore")

with contextlib.redirect_stderr(io.StringIO()):
    os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
    import litellm
    litellm.suppress_debug_info = True
    import logging
    logging.getLogger("LiteLLM").setLevel(logging.WARNING)

    # Core ADK
    from google.adk.agents import LlmAgent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.adk.models.lite_llm import LiteLlm
    from google.genai import types

import asyncio
import uuid

print("✅ Imports successful.")

✅ Imports successful.


# The Four Building Blocks

Most frameworks throw a dozen concepts at you in module one. ADK has four that matter on day one:

| Building block | What it is | What you do with it |
|---|---|---|
| `LlmAgent` | An LLM wired to instructions, a model, and (optionally) tools | You define one |
| `Runner` | The loop that drives a conversation | You call `.run_async()` |
| `Event` | Every message, tool call, tool response, state change | You read them to see what's happening |
| `Session` | The conversation's memory — history plus a state dict | You create one per conversation |

Read the table against your hand-rolled loop: the `Runner` is your Steps 1–5 written once and properly, `Session` is your `messages` list with a real home, and every `print()` you sprinkled becomes a typed `Event`. Everything else in this course composes on top of these four.

# Your First Agent

Four required arguments: a name, a model, a description, and an instruction. No tools yet — this is just an LLM with a system prompt, wrapped in ADK's plumbing.

One thing to notice: the next cell makes **no API call**. `LlmAgent(...)` only builds a Python object that *describes* the agent. Nothing talks to OpenRouter until we run it — which is exactly why we'll need a Runner in a moment.

In [6]:
greeter = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Greets the user in a friendly way.",
    instruction="You are a friendly greeter. Respond in one short sentence.",
)

print(f"✅ Agent '{greeter.name}' built.")

✅ Agent 'greeter' built.


### The same agent on native Gemini — for comparison

Because ADK speaks Gemini natively, on Gemini the cell above would lose the wrapper and take a plain string:

```python
greeter = LlmAgent(
    name="greeter",
    model="gemini-2.5-flash",   # plain string — ADK talks to Gemini natively
    description="Greets the user in a friendly way.",
    instruction="You are a friendly greeter. Respond in one short sentence.",
)
```

That is the *entire* difference — everything downstream (Runner, Session, Events, tools) is identical. We stay on `LiteLlm` throughout Part 1 to prove ADK isn't tied to Google's models; Part 2 (M11–M13) switches to native Gemini for the capabilities only Gemini unlocks. No Google key needed today.

## Running It — Why a Runner?

*"Until now I just called functions. Why do I need a Runner?"*

Because an agent is a **loop**, not a single call: the model answers → maybe it asks for a tool → the tool runs → the result goes back → the model answers again… until there's a final answer. Somebody has to run that loop, feed it the growing history, and record what happened. That somebody is the **Runner** — your Steps 1–5, written once, with all the bookkeeping.

*"And a Session?"*

The Runner needs to know *which* conversation it is continuing. A **Session** is one conversation: its message history plus a small state dict. `InMemorySessionService` keeps sessions in a Python dict — gone when the kernel restarts, which is fine for learning.

## Two Rules of `async` (Your First Encounter)

Most of an agent's life is waiting — for the model's HTTP response, for a tool's API call. `async` is Python's way of saying: *"this function waits on something external; while it waits, let other work run."* ADK is written async-first because a real agent server juggles many conversations at once.

For us, in a notebook, two rules are enough:

1. A function that uses `await` inside must be declared `async def`.
2. You call such a function with `await` in front: `await chat(...)`, not `chat(...)`.

Jupyter and Colab let you write `await` directly in a cell. That is genuinely all you need — you don't have to understand the event loop to use ADK.

## And What Does Running Return? A Stream

`runner.run_async()` doesn't return one answer. It returns a *stream*: you loop over it with `async for`, and it hands you an **Event** for every step — text, tool call, tool result — the moment it happens. For a simple question that's a single event; add tools and you'll watch the stream grow.

The next cell builds our `chat()` helper — we'll reuse it all course long. Here is what its lines do, in plain words:

### The helper, line by line

Five lines in `chat()` do real work — the rest is pretty-printing:

- `create_session(...)` — open a fresh folder for this one conversation; its history and state will live there.
- `Runner(agent=..., app_name=..., session_service=...)` — hire the operator: *which* agent to run and *where* the session folders live.
- `types.Content(role="user", parts=[types.Part(text=prompt)])` — the envelope format models use for messages: "a user said this text". Every event you'll ever read carries this same Content-with-Parts shape.
- `async for event in runner.run_async(...)` — start the loop and watch events arrive one by one.
- The `if p.text / p.function_call / p.function_response` block — just printing: is this event text, a tool call, or a tool result?

That's all `chat()` is: open a folder, hire the operator, hand over the envelope, print what happens. It looks long because of the printing, not the logic.

In [7]:
APP = "m01_demo"
USER = "student"

session_service = InMemorySessionService()

async def chat(agent, prompt: str):
    """Send one user message to the agent and print every event it emits."""
    # Fresh session per call so previous runs don't leak in.
    sid = f"session-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)

    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    print(f"USER: {prompt}\n")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        tag = "[FINAL]" if event.is_final_response() else "[step]"
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    print(f"{tag} {event.author}: {p.text.strip()}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args)})")
                if p.function_response:
                    print(f"[tool_resp] {p.function_response.response}")

await chat(greeter, "Hi, what's your name?")

USER: Hi, what's your name?



[FINAL] greeter: Hi! I’m Greeter, nice to meet you.


### 🔍 What just happened?

One event came back, carrying the final text — the minimum a conversation can produce. **Events are ADK's unit of observability**: every message, tool call, tool response or hand-off is one. In the next section we add a tool, and the stream grows from one event to three.

# Add a Tool — The Docstring Is the Schema

This is the best row of the mapping table, made real. In the bridge you passed a hand-written 34-line schema *plus* the function. In ADK you pass **just the function** — the schema is generated from two pieces of Python you already write out of habit:

- **Type hints** (`location: str`) — ignored by Python at runtime, but ADK reads them to tell the model *"this argument is a string"*.
- **The docstring** — normally documentation for humans; here it is sent to the model as the tool's description. The model decides *whether and how* to call your tool purely from this text — so write it for the model, not for a code reviewer.

The next cell repeats the bridge's function unchanged, then prints the exact schema ADK generates from it. Seeing is believing.

In [8]:
# The SAME function from the bridge, repeated unchanged — read the type hints
# and the docstring here, right next to the explanation above.
def get_current_weather(location: str, format: str) -> dict:
    """Get the current weather for a location.

    Args:
        location (str): The city, e.g. "Bratislava" or "Prague".
        format (str): The temperature format, either 'celsius' or 'fahrenheit'.

    Returns:
        dict: location, temperature, unit and condition (mock data).
    """
    city = location.split(",")[0].strip()
    condition, celsius = FAKE_WEATHER.get(city, ("Unknown", None))
    if celsius is None:
        return {"location": location, "condition": "Unknown", "error": f"No data for {location}."}
    temp = celsius if format == "celsius" else round(celsius * 9 / 5 + 32, 1)
    return {"location": location, "temperature": temp, "unit": format, "condition": condition}

# What does ADK actually build from those two pieces? FunctionTool is the wrapper ADK
# puts around any plain function; _get_declaration() is the schema it sends to the model.
# (A private helper — fine for peeking, not something you call in real code.)
from google.adk.tools.function_tool import FunctionTool

declaration = FunctionTool(get_current_weather)._get_declaration()
print("name:       ", declaration.name)
print("description:", declaration.description)
print("parameters:")
print(json.dumps(declaration.parameters_json_schema, indent=2))

name:        get_current_weather
description: Get the current weather for a location.

Args:
    location (str): The city, e.g. "Bratislava" or "Prague".
    format (str): The temperature format, either 'celsius' or 'fahrenheit'.

Returns:
    dict: location, temperature, unit and condition (mock data).
parameters:
{
  "properties": {
    "location": {
      "title": "Location",
      "type": "string"
    },
    "format": {
      "title": "Format",
      "type": "string"
    }
  },
  "required": [
    "location",
    "format"
  ],
  "title": "get_current_weatherParams",
  "type": "object"
}


### 🔍 Compare with your hand-written schema

Same `name`, same parameter types, same `required` list as your 34-line dict — and this time you wrote none of it. The docstring travels as the `description`; the type hints became the `type` fields.

One honest note: ADK 2.7 sends the docstring as a single block (it does not split the `Args:` section into per-parameter descriptions), so make sure it reads well as one piece.

Now hand the function to an agent — and notice what is *absent* from the next cell: no schema dict, no dispatch dictionary.

In [9]:
weather_agent = LlmAgent(
    name="weather_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports current weather for a given location.",
    instruction=(
        "You are a weather assistant. When the user asks about weather, "
        "call the get_current_weather tool and report what it returns. Be brief."
    ),
    tools=[get_current_weather],   # just the function — no schema dict, no dispatch dict
)

await chat(weather_agent, "What's the weather in Prague right now? Use Celsius.")


USER: What's the weather in Prague right now? Use Celsius.



[tool_call] get_current_weather({'location': 'Prague', 'format': 'celsius'})
[tool_resp] {'location': 'Prague', 'temperature': 14.0, 'unit': 'celsius', 'condition': 'Cloudy'}


[FINAL] weather_agent: Prague is currently **14°C and cloudy**.


### 🔍 What just happened — three events

1. `[tool_call] get_current_weather({'location': 'Prague', 'format': 'celsius'})` — the model decided a tool was needed.
2. `[tool_resp] {...}` — ADK executed your function and fed the result back.
3. `[FINAL]` — the model wrote the answer from the tool's output.

All visible, all inspectable. If the model had called the tool wrong, you'd see it; if it had refused, you'd see that too. This visibility is ADK's biggest advantage over the loop from the bridge — there, your `print()`s were the only window.

### 🎯 Mini-tasks

1. **Break the tool on purpose.** Ask the weather agent about `"Reykjavik"` (not in `FAKE_WEATHER`). Read the events — does the model pass the error through honestly, or hallucinate a forecast?
2. **Bring back the forecast.** Add `get_n_day_weather_forecast(location, format, num_days)` from the previous course (invent fake data), add it to `tools=[...]`, and ask for a 3-day forecast in Munich. Back then it cost you a second 30-line schema — count the lines you write now.

## A Word on `adk web`

ADK also installs a command-line tool, and `adk web` is a free visual debugger for your agents. It expects the agent in a small folder package:

```
my_agents/
└── weather_agent/
    ├── __init__.py     # one line:  from . import agent
    └── agent.py        # the LlmAgent from above, assigned to a variable named  root_agent
```

Run `adk web my_agents` from a terminal (key in a `.env` there), open `http://localhost:8000`, pick your agent — and you get the same event stream we just printed, as a clickable timeline with the full JSON of every event. We stay in notebooks for the videos (they record better), but at home this is the best debugger you get for free. M10 reuses the same folder layout for deployment.

# Key Takeaways

- **You had already built an agent.** ADK replaces your schema (the docstring does it), your dispatch dict (built-in), your loop (`Runner`), and your `messages` list (`Session`).
- An `LlmAgent` is a *description* of an agent; the `Runner` makes it talk, yielding an `Event` per step.
- `async` in this course is two rules: `async def` when a function awaits inside; `await` when you call it.
- The `LiteLlm` adapter is why the same agent runs on GPT, Claude, Gemini or Qwen — the course default is deliberately not a Google model.
- Carry the four building blocks forward: **Agent · Runner · Event · Session**.

# Next up — Module 02

One flavor of tool — the plain function — is now yours. ADK has three more: a whole REST API, an MCP server, another agent. Module 02 builds one of each, around a single story: an IT help desk.